In [2]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.8/151.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 16.6 MB/s eta 0:00:00


In [3]:
import ccxt

exchange = ccxt.coinbase({'enableRateLimit': True})

def inventory_adjusted_quote(mid_price, inventory_pct, max_spread=0.001):
    """
    库存调整的做市商报价

    参数:
        mid_price: 当前中间价
        inventory_pct: 库存偏离度(-100~+100, 正=手里BTC太多)
        max_spread: 基础价差(默认0.1%)

    返回:
        调整后的买单和卖单报价
    """
    # 偏移量：库存越高，双向报价都往下偏
    skew = inventory_pct / 100 * max_spread

    bid = mid_price * (1 - max_spread/2 - skew)
    ask = mid_price * (1 + max_spread/2 - skew)

    return {
        "bid": round(bid, 2),
        "ask": round(ask, 2),
        "spread_bp": round((ask - bid) / bid * 10000, 2),
        "skew_bp": round(skew * 10000, 2)
    }

# ===== 三个场景对比 =====

mid = 63000

# 场景1：库存中性
q0 = inventory_adjusted_quote(mid, 0)
print(f"平衡:  买${q0['bid']} | 卖${q0['ask']} | 价差{q0['spread_bp']}bp")

# 场景2：BTC太多（+50%）
q1 = inventory_adjusted_quote(mid, 50)
print(f"多仓:  买${q1['bid']} | 卖${q1['ask']} | 偏移{q1['skew_bp']}bp")

# 场景3：BTC太少（-50%）
q2 = inventory_adjusted_quote(mid, -50)
print(f"空仓:  买${q2['bid']} | 卖${q2['ask']} | 偏移{q2['skew_bp']}bp")

平衡:  买$62968.5 | 卖$63031.5 | 价差10.01bp
多仓:  买$62937.0 | 卖$63000.0 | 偏移5.0bp
空仓:  买$63000.0 | 卖$63063.0 | 偏移-5.0bp
